In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.backends.cudnn.benchmark = True  # speed up fixed-size convolutions

%matplotlib inline

In [3]:
DATA_DIR = "data/art_portraits/Portraits_update"

image_size = 32
batch_size = 128          # ⬆ doubled from 64 → better GPU utilization
num_workers = 6

# DCGAN normalization ([-1, 1])
stats = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
full_ds = ImageFolder(
    DATA_DIR,
    transform=T.Compose([
        T.Resize(image_size),
        T.CenterCrop(image_size),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
        T.Normalize(*stats)
    ])
)

# Subsample to 3000 images for faster training
np.random.seed(42)
indices = np.random.choice(len(full_ds), size=3000, replace=False)
train_ds = Subset(full_ds, indices)

train_dl = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

print("Total images:", len(train_ds))

Total images: 3000


In [5]:
images, _ = next(iter(train_dl))
images = images.to(device)

print("Batch shape:", images.shape)
print("Min / Max:", images.min().item(), images.max().item())


Batch shape: torch.Size([128, 3, 32, 32])
Min / Max: -1.0 1.0


In [6]:
nz = 100      # latent vector size
ngf = 64      # generator feature maps
ndf = 64      # discriminator feature maps
nc = 3        # number of channels (RGB)


In [10]:
def weights_init(m):
    classname = m.__class__.__name__

    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)

    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [11]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(

            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf * 2, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)

In [12]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(

            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.net(x).view(-1)

In [13]:
netG = Generator().to(device)
netD = Discriminator().to(device)

# Apply DCGAN weight init
netG.apply(weights_init)
netD.apply(weights_init)

print(netG)
print(netD)

Generator(
  (net): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 3, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): Tanh()
  )
)
Discriminator(
  (net): Sequential(
    (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4

In [14]:
# BCEWithLogitsLoss = Sigmoid + BCE fused (numerically stable, AMP-friendly)
criterion = nn.BCEWithLogitsLoss()

lr = 0.0002
beta1 = 0.5

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# AMP scaler for mixed-precision training
scaler = torch.amp.GradScaler("cuda")

# Fixed noise for monitoring generator progression
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Logging
G_losses = []
D_losses = []

# Output directories
os.makedirs("outputs", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

In [15]:
num_epochs = 300

real_scores = []
fake_scores = []

best_d_loss = float("inf")   

for epoch in range(num_epochs):

    netG.train()
    netD.train()

    epoch_loss_g = 0.0
    epoch_loss_d = 0.0
    epoch_real_score = 0.0
    epoch_fake_score = 0.0

    for i, (real_images, _) in enumerate(
        tqdm(train_dl, desc=f"Epoch [{epoch+1}/{num_epochs}]")
    ):
        real_images = real_images.to(device, non_blocking=True)
        b_size = real_images.size(0)

        if epoch == 0 and i == 0:
            print("CUDA available:", torch.cuda.is_available())
            print("Tensor device:", real_images.device)
            print("GPU memory:",
                  torch.cuda.memory_allocated() // (1024**2), "MB")

        
        real_labels = torch.full((b_size,), 0.9, device=device)
        fake_labels = torch.zeros(b_size, device=device)

       
        netD.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            real_output = netD(real_images).view(-1)
            errD_real = criterion(real_output, real_labels)

            noise = torch.randn(b_size, nz, 1, 1, device=device)
            fake_images = netG(noise)
            fake_output = netD(fake_images.detach()).view(-1)
            errD_fake = criterion(fake_output, fake_labels)

            errD = errD_real + errD_fake

        scaler.scale(errD).backward()
        scaler.step(optimizerD)

        real_score = torch.sigmoid(real_output).mean().item()
        fake_score = torch.sigmoid(fake_output).mean().item()

        # =============================
        # (2) Train Generator
        # =============================
        netG.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            fake_output = netD(fake_images).view(-1)
            errG = criterion(fake_output, torch.ones(b_size, device=device))

        scaler.scale(errG).backward()
        scaler.step(optimizerG)
        scaler.update()

        # =============================
        # Logging
        # =============================
        epoch_loss_d += errD.item()
        epoch_loss_g += errG.item()
        epoch_real_score += real_score
        epoch_fake_score += fake_score

    # -----------------------------
    # Epoch averages
    # -----------------------------
    n_batches = len(train_dl)

    epoch_loss_d /= n_batches
    epoch_loss_g /= n_batches
    epoch_real_score /= n_batches
    epoch_fake_score /= n_batches

    D_losses.append(epoch_loss_d)
    G_losses.append(epoch_loss_g)
    real_scores.append(epoch_real_score)
    fake_scores.append(epoch_fake_score)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"loss_g: {epoch_loss_g:.4f}, "
        f"loss_d: {epoch_loss_d:.4f}, "
        f"real_score: {epoch_real_score:.4f}, "
        f"fake_score: {epoch_fake_score:.4f}"
    )

    # -----------------------------
    # Save generated images
    # -----------------------------
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        save_image(fake, f"outputs/DCGAN-{epoch+1:04d}.png",
                   nrow=8, normalize=True)

    # -----------------------------
    # Save BEST discriminator (TRICK)
    # -----------------------------
    if epoch_loss_d < best_d_loss:
        best_d_loss = epoch_loss_d
        torch.save(netD.state_dict(), "checkpoints/best_dcgan_discriminator.pth")
        print("Saved new best discriminator")

# -----------------------------
# Final model save
# -----------------------------
torch.save(netG.state_dict(), "checkpoints/dcgan_generator_final.pth")
torch.save(netD.state_dict(), "checkpoints/dcgan_discriminator_final.pth")

Epoch [1/300]:   0%|          | 0/24 [00:00<?, ?it/s]

CUDA available: True
Tensor device: cuda:0
GPU memory: 18 MB


Epoch [1/300]: 100%|██████████| 24/24 [00:28<00:00,  1.18s/it]


Epoch [1/300] loss_g: 3.3344, loss_d: 1.4774, real_score: 0.6587, fake_score: 0.4773
Saved new best discriminator


Epoch [2/300]: 100%|██████████| 24/24 [00:27<00:00,  1.15s/it]


...

Epoch [298/300] loss_g: 2.6857, loss_d: 0.6152, real_score: 0.7544, fake_score: 0.1514


Epoch [299/300]: 100%|██████████| 24/24 [00:26<00:00,  1.11s/it]


Epoch [299/300] loss_g: 2.6671, loss_d: 0.5980, real_score: 0.7596, fake_score: 0.1452


Epoch [300/300]: 100%|██████████| 24/24 [00:27<00:00,  1.13s/it]

Epoch [300/300] loss_g: 2.9739, loss_d: 1.5028, real_score: 0.6357, fake_score: 0.2653


In [ ]:
torch.save(netD.state_dict(), "streamlit_app/models/dcgan_discriminator.pth")
torch.save(netG.state_dict(), "streamlit_app/models/dcgan_generator.pth")